In [17]:
import requests
import pandas as pd
from collections import defaultdict

In [4]:
# Step 2: Fetch all the indicator available on adapta
url = "https://sistema.adaptabrasil.mcti.gov.br/api/hierarquia/adaptabrasil"
response = requests.get(url)

if response.status_code == 200:
    data = response.json()
else:
    raise Exception(f"Failed to fetch data: {response.status_code}")

df = pd.json_normalize(data)
df.head(5)

,id,name,title,shortname,simple_description,complete_description,equation,level,pessimist,indicator_id_master,...,menu_structure.defaultclippingresolution.resolution,menu_structure.defaultclippingresolution.table,menu_structure.defaultclippingresolution.label,menu_structure.defaultclippingresolution.resolution_order,menu_structure.defaultclippingresolution.recordcount,menu_structure.defaultclippingresolution.default,menu_structure.defaultclippingresolution.max_number_of_records,menu_structure.defaultclippingresolution.group_id,menu_structure.clippingresolutions,legend.items
0,0,Base conceitual conforme definições do IPCC,Risco Climático,Risco Climático,Base conceitual conforme definições do IPCC,"<html><p><img src=""https://s3.us-east-1.amazon...",None,0,NaN,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,Recursos hídricos,Impactos para recursos hídricos,Recursos hídricos,Consequências esperadas e resultantes das muda...,"São os efeitos sobre vidas, meios de subsistên...",None,1,NaN,0,...,Município,county,Municípios,5.0,5570.0,1.0,980.0,1.0,"[{'id': 'brasil', 'category': 'Brasil', 'order...",NaN
2,5000,Segurança alimentar,Impactos para segurança alimentar,Segurança alimentar,Consequências esperadas e resultantes das muda...,"Os impactos sobre vidas, meios de subsistência...",None,1,1.0,0,...,Município,county,Municípios,5.0,5570.0,1.0,980.0,1.0,"[{'id': 'brasil', 'category': 'Brasil', 'order...","[{'order': 1, 'label': 'Muito baixo', 'color':..."
3,10000,Segurança energética,Impactos para segurança energética,Segurança energética,Consequências esperadas e resultantes das muda...,"São os efeitos sobre vidas, meios de subsistên...",None,1,1.0,0,...,Município,county,Municípios,5.0,5570.0,1.0,980.0,1.0,"[{'id': 'brasil', 'category': 'Brasil', 'order...","[{'order': 1, 'label': 'Muito baixo', 'color':..."
4,40000,Infraestrutura portuária,Impactos para infraestrutura portuária,Infraestrutura portuária,Consequências esperadas e resultantes das muda...,"São os efeitos sobre vidas, meios de subsistên...",None,1,1.0,0,...,Porto,harbor,Portos,1.0,21.0,1.0,21.0,2.0,"[{'id': 'brasil', 'category': 'Brasil', 'order...","[{'order': 1, 'label': 'Muito baixo', 'color':..."


In [19]:
df.to_csv('./data/adapta_indicators_ids.csv', index=False)

In [ ]:
#downloads the data from adapta, this take 30 minutes to run so it's commented out
url = "https://sistema.adaptabrasil.mcti.gov.br/api/hierarquia/adaptabrasil"
response = requests.get(url)
if response.status_code == 200:
    data = response.json()
else:
    raise Exception(f"Failed to fetch data: {response.status_code}")

# Normalize the JSON data to a DataFrame
df = pd.json_normalize(data)

# List to hold all API responses
all_responses = []

# Get the 9th and 10th rows (note: iloc is zero-indexed)
rows_to_process = df#.iloc[8:10]  # This slice gets rows with index 8 and 9, which are the 9th and 10th rows

# Process each selected row
for _, row in rows_to_process.iterrows():
    identifier = row.get('id')
    years = row.get('years')  # Can be None, int, str, or an iterable of mixed values

    year_list = []
    if years is not None:
        # Normalize to an iterable so we can handle scalar and array-like inputs uniformly.
        values = years if isinstance(years, (list, tuple, set)) else [years]

        for item in values:
            if item is None:
                continue
            if isinstance(item, (int, float)):
                year_list.append(int(item))
                continue

            # Strings may hold one or multiple comma-separated years.
            for token in str(item).split(','):
                token = token.strip()
                if token:
                    year_list.append(int(token))

        # Remove duplicates while preserving order.
        year_list = list(dict.fromkeys(year_list))

    # Now process each year in year_list
    for year in year_list:
            if year > 2024:
                # Construct URLs for years greater than 2024
                for scenario in (1, 2):
                    url = f"https://sistema.adaptabrasil.mcti.gov.br/api/mapa-dados/BR/municipio/{identifier}/{year}/{scenario}/adaptabrasil"
                    response = requests.get(url)
                    if response.status_code == 200:
                        data = response.json()  # Parse the JSON response
                        df_data = pd.json_normalize(data)  # Convert to DataFrame
                        df_data['indicator_id'] = identifier  # Add the indicator ID for reference
                        df_data['year'] = year  # Add the year for reference
                        all_responses.append(df_data)  # Append the fetched data to the list
                    else:
                        print(f"Failed to fetch data for ID {identifier}, Year {year}, URL: {url}")
            else:
                # Construct URL for years less than or equal to 2024
                url = f"https://sistema.adaptabrasil.mcti.gov.br/api/mapa-dados/BR/municipio/{identifier}/{year}/null/adaptabrasil"
                response = requests.get(url)
                if response.status_code == 200:
                    data = response.json()  # Parse the JSON response
                    df_data = pd.json_normalize(data)  # Convert to DataFrame
                    df_data['indicator_id'] = identifier  # Add the indicator ID for reference
                    df_data['year'] = year  # Add the year for reference
                    all_responses.append(df_data)  # Append the fetched data to the list
                else:
                    print(f"Failed to fetch data for ID {identifier}, Year {year}, URL: {url}")

# Concatenate all DataFrames in the list into a single DataFrame if there are responses
if all_responses:
    combined_df = pd.concat(all_responses, ignore_index=True)
    print(combined_df)  # Check out the combined DataFrame
else:
    print("No responses fetched or the response list is empty.")


combined_df.to_csv('.data/adapta_city_data.csv', index=False)

             id geocod_ibge                    name  indicator_id  year  \
0        2644.0     5200050      Abadia de Goiás/GO             2  2020   
1         754.0     3100104  Abadia dos Dourados/MG             2  2020   
2        2645.0     5200100            Abadiânia/GO             2  2020   
3         755.0     3100203               Abaeté/MG             2  2020   
4        2938.0     1500107           Abaetetuba/PA             2  2020   
...         ...         ...                     ...           ...   ...   
2105455  4869.0     2933604          Xique-Xique/BA         90034  2017   
2105456  4201.0     2517407               Zabelê/PB         90034  2017   
2105457  2376.0     3557154             Zacarias/SP         90034  2017   
2105458  3403.0     2114007              Zé Doca/MA         90034  2017   
2105459   126.0     4219853               Zortéa/SC         90034  2017   

        scenario_id  pessimist  value valuecolor   rangelabel  
0              None        1.0   0.